# 4-D Sweep Visualization — E-I Balanced Network

Interactive exploration of the parameter sweep over **N_total × g × epsilon × J_mV**.

- Grey hatched cells = **no spiking activity** (neurons never reached threshold → metrics are NaN)
- Use the dropdown in each section to pick which metric to inspect

In [ ]:
import json, os, glob, math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from IPython.display import display

%matplotlib inline

# ── Load data ──
EXPERIMENT_DIR = os.path.dirname(os.path.abspath('__file__'))
# Try both possible locations
for candidate in [
    os.path.join(EXPERIMENT_DIR, 'results', 'sweep_4d_ai.jsonl'),
    os.path.join(os.getcwd(), 'results', 'sweep_4d_ai.jsonl'),
]:
    matches = sorted(glob.glob(candidate))
    if matches:
        log_path = matches[-1]
        break
else:
    raise FileNotFoundError('No sweep_4d_ai.jsonl found')

records = []
with open(log_path) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Loaded {len(records)} records from {log_path}')

# ── Discover grid axes ──
def axis_vals(key):
    return np.array(sorted({r['params'][key] for r in records}))

N_vals   = axis_vals('N_total')
J_vals   = axis_vals('J_mV')
g_vals   = axis_vals('g')
eps_vals = axis_vals('epsilon')
n_N, n_J, n_g, n_eps = len(N_vals), len(J_vals), len(g_vals), len(eps_vals)

# Lookup: (N, J, g, eps) → metrics dict
lookup = {}
for r in records:
    p = r['params']
    lookup[(p['N_total'], p['J_mV'], p['g'], p['epsilon'])] = r['metrics']

print(f'Grid: {n_N} N × {n_J} J × {n_g} g × {n_eps} eps')
print(f'  N_total : {N_vals}')
print(f'  J_mV    : {J_vals}')
print(f'  g       : {g_vals}')
print(f'  epsilon : {eps_vals}')

# Count activity
n_active = sum(1 for r in records if r['metrics']['fr_E'] > 0)
n_silent = len(records) - n_active
print(f'\nActive (fr_E > 0): {n_active}/{len(records)}  |  Silent: {n_silent}/{len(records)}')

In [ ]:
# ── Helper functions ──

def _edges(v):
    """Cell edges for pcolormesh so ticks land on centres."""
    v = np.asarray(v, dtype=float)
    if len(v) == 1:
        return np.array([v[0] - 0.5, v[0] + 0.5])
    d = np.diff(v) / 2.0
    return np.concatenate([[v[0] - d[0]], v[:-1] + d, [v[-1] + d[-1]]])

G_edges   = _edges(g_vals)
EPS_edges = _edges(eps_vals)


def slice_2d(metric, N, J):
    """(n_eps × n_g) array for one (N, J) panel."""
    Z = np.full((n_eps, n_g), np.nan)
    for ie, e in enumerate(eps_vals):
        for ig, g in enumerate(g_vals):
            m = lookup.get((N, J, g, e))
            if m is not None:
                Z[ie, ig] = m[metric]
    return Z


def silent_mask_2d(N, J):
    """Boolean mask: True where fr_E == 0 (no activity)."""
    mask = np.zeros((n_eps, n_g), dtype=bool)
    for ie, e in enumerate(eps_vals):
        for ig, g in enumerate(g_vals):
            m = lookup.get((N, J, g, e))
            if m is None or m['fr_E'] == 0.0:
                mask[ie, ig] = True
    return mask


def _safe_twoslope(Z, center):
    vmin, vmax = np.nanmin(Z), np.nanmax(Z)
    if np.isnan(vmin):
        vmin, vmax = center - 1, center + 1
    vmin = min(vmin, center - 1e-6)
    vmax = max(vmax, center + 1e-6)
    return mcolors.TwoSlopeNorm(vcenter=center, vmin=vmin, vmax=vmax)


def draw_hatching(ax, mask, g_edges, eps_edges):
    """Draw grey hatched rectangles over silent cells."""
    for ie in range(mask.shape[0]):
        for ig in range(mask.shape[1]):
            if mask[ie, ig]:
                rect = Rectangle(
                    (g_edges[ig], eps_edges[ie]),
                    g_edges[ig+1] - g_edges[ig],
                    eps_edges[ie+1] - eps_edges[ie],
                    linewidth=0.5, edgecolor='#888',
                    facecolor='#d0d0d0', hatch='///', zorder=2)
                ax.add_patch(rect)

print('Helpers ready.')

## 1. Firing Rate (E population)
Each sub-panel is one (N, J) pair. Axes within each panel: **g** (x) × **epsilon** (y).  
Grey hatched = no activity.

In [ ]:
def plot_metric_grid(metric, title, cmap, norm_fn=None, clamp=None):
    """Grid-of-grids figure for one metric with silent-cell hatching."""
    Z_all = np.concatenate([
        slice_2d(metric, N, J).ravel() for N in N_vals for J in J_vals
    ])
    valid = Z_all[~np.isnan(Z_all)]

    fig, axes = plt.subplots(
        n_N, n_J, figsize=(3.4 * n_J, 2.8 * n_N), squeeze=False)
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)

    for iN, N in enumerate(N_vals):
        for iJ, J in enumerate(J_vals):
            ax = axes[iN, iJ]
            Z = slice_2d(metric, N, J)
            mask = silent_mask_2d(N, J)
            if clamp:
                Z = np.clip(Z, *clamp)
            norm = None
            if norm_fn and len(valid) > 0:
                norm = norm_fn(valid if not clamp else np.clip(valid, *clamp))
            ax.pcolormesh(G_edges, EPS_edges, Z, cmap=cmap,
                          norm=norm, shading='flat')
            draw_hatching(ax, mask, G_edges, EPS_edges)
            ax.set_xticks(g_vals)
            ax.set_yticks(eps_vals)
            ax.tick_params(labelsize=7)
            ax.set_title(f'N={int(N)}  J={J}', fontsize=8, pad=3)
            if iN == n_N - 1:
                ax.set_xlabel('g', fontsize=9)
            else:
                ax.set_xticklabels([])
            if iJ == 0:
                ax.set_ylabel('ε', fontsize=9)
            else:
                ax.set_yticklabels([])

    # Colour bar
    fig.subplots_adjust(right=0.86, hspace=0.35, wspace=0.20)
    cbar_ax = fig.add_axes([0.88, 0.15, 0.015, 0.7])
    if norm_fn and len(valid) > 0:
        sm = plt.cm.ScalarMappable(
            cmap=cmap, norm=norm_fn(valid if not clamp else np.clip(valid, *clamp)))
    else:
        sm = plt.cm.ScalarMappable(cmap=cmap)
        if len(valid) > 0:
            sm.set_clim(np.nanmin(valid), np.nanmax(valid))
    sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax)

    # Legend for hatching
    legend_patch = mpatches.Patch(
        facecolor='#d0d0d0', edgecolor='#888', hatch='///',
        label='No activity (fr = 0 Hz — below threshold)')
    fig.legend(handles=[legend_patch], loc='lower center',
               fontsize=9, ncol=1, bbox_to_anchor=(0.44, -0.02),
               frameon=True, fancybox=True)
    plt.show()
    return fig

_ = plot_metric_grid('fr_E', 'Firing rate — E (Hz)', 'viridis')

## 2. CV(ISI) — E population
Coefficient of variation of inter-spike intervals. AI regime expects CV ≈ 1.

In [ ]:
_ = plot_metric_grid('cv_E', 'CV(ISI) — E', 'RdBu_r',
    norm_fn=lambda Z: _safe_twoslope(np.clip(Z, 0, 2), 1.0), clamp=(0, 2))

## 3. Mean Pairwise Correlation — E population
AI regime expects near-zero correlation.

In [ ]:
_ = plot_metric_grid('corr_E', 'Mean pairwise correlation — E', 'RdBu_r',
    norm_fn=lambda Z: _safe_twoslope(Z, 0.0))

## 4. Fano Factor — E population
AI regime expects Fano ≈ 1.

In [ ]:
_ = plot_metric_grid('fano_E', 'Fano factor — E (10 ms bins)', 'RdBu_r',
    norm_fn=lambda Z: _safe_twoslope(np.clip(Z, 0, 3), 1.0), clamp=(0, 3))

## 5. Composite AI-regime Score
Score = product of sub-scores for CV≈1, |Corr|≈0, Fano≈1.  
**Bright = close to AI regime. Dark = far from AI.**  
Silent cells (no spikes) score **0** and are hatched.

In [ ]:
CV_TOL, CORR_TOL, FANO_TOL = 0.5, 0.05, 0.5

def ai_score(N, J):
    cv   = slice_2d('cv_E',   N, J)
    corr = slice_2d('corr_E', N, J)
    fano = slice_2d('fano_E', N, J)
    s_cv   = 1 - np.clip(np.abs(cv   - 1.0) / CV_TOL,   0, 1)
    s_corr = 1 - np.clip(np.abs(corr - 0.0) / CORR_TOL, 0, 1)
    s_fano = 1 - np.clip(np.abs(fano - 1.0) / FANO_TOL, 0, 1)
    score = s_cv * s_corr * s_fano
    # Silent cells → score 0
    score[np.isnan(score)] = 0.0
    return score

fig_ai, axes_ai = plt.subplots(
    n_N, n_J, figsize=(3.4 * n_J, 2.8 * n_N), squeeze=False)
fig_ai.suptitle(
    f'AI-regime score (E)\n'
    f'CV≈1 ± {CV_TOL}    |Corr| ≤ {CORR_TOL}    Fano≈1 ± {FANO_TOL}',
    fontsize=13, fontweight='bold', y=1.04)

for iN, N in enumerate(N_vals):
    for iJ, J in enumerate(J_vals):
        ax = axes_ai[iN, iJ]
        Z = ai_score(N, J)
        mask = silent_mask_2d(N, J)
        ax.pcolormesh(G_edges, EPS_edges, Z, cmap='magma',
                      shading='flat', vmin=0, vmax=1)
        draw_hatching(ax, mask, G_edges, EPS_edges)
        ax.set_xticks(g_vals)
        ax.set_yticks(eps_vals)
        ax.tick_params(labelsize=7)
        ax.set_title(f'N={int(N)}  J={J}', fontsize=8, pad=3)
        if iN == n_N - 1:
            ax.set_xlabel('g', fontsize=9)
        else:
            ax.set_xticklabels([])
        if iJ == 0:
            ax.set_ylabel('ε', fontsize=9)
        else:
            ax.set_yticklabels([])

fig_ai.subplots_adjust(right=0.86, hspace=0.35, wspace=0.20)
cbar_ax = fig_ai.add_axes([0.88, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='magma', norm=mcolors.Normalize(0, 1))
sm.set_array([])
fig_ai.colorbar(sm, cax=cbar_ax)

legend_patch = mpatches.Patch(
    facecolor='#d0d0d0', edgecolor='#888', hatch='///',
    label='No activity (fr = 0 Hz — below threshold)')
fig_ai.legend(handles=[legend_patch], loc='lower center',
              fontsize=9, ncol=1, bbox_to_anchor=(0.44, -0.02),
              frameon=True, fancybox=True)
plt.show()

## 6. Single-panel Deep Dive
Pick a specific (N, J) pair and see all metrics + activity mask side by side.

In [ ]:
# ── CHANGE THESE to explore different slices ──
PICK_N = 2500
PICK_J = 0.4

metrics_info = [
    ('fr_E',   'Firing rate E (Hz)',  'viridis', None,    None),
    ('fr_I',   'Firing rate I (Hz)',  'viridis', None,    None),
    ('cv_E',   'CV(ISI) E',          'RdBu_r',  1.0,    (0, 2)),
    ('cv_I',   'CV(ISI) I',          'RdBu_r',  1.0,    (0, 2)),
    ('corr_E', 'Corr E',             'RdBu_r',  0.0,    None),
    ('corr_I', 'Corr I',             'RdBu_r',  0.0,    None),
    ('fano_E', 'Fano E',             'RdBu_r',  1.0,    (0, 3)),
    ('fano_I', 'Fano I',             'RdBu_r',  1.0,    (0, 3)),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8), squeeze=False)
fig.suptitle(f'All metrics — N={PICK_N}, J={PICK_J}', fontsize=14, fontweight='bold')

mask = silent_mask_2d(PICK_N, PICK_J)

for idx, (mkey, mlabel, cmap, center, clamp) in enumerate(metrics_info):
    ax = axes[idx // 4, idx % 4]
    Z = slice_2d(mkey, PICK_N, PICK_J)
    if clamp:
        Z = np.clip(Z, *clamp)
    valid = Z[~np.isnan(Z)]

    norm = None
    if center is not None and len(valid) > 0:
        norm = _safe_twoslope(valid, center)

    pcm = ax.pcolormesh(G_edges, EPS_edges, Z, cmap=cmap,
                        norm=norm, shading='flat')
    draw_hatching(ax, mask, G_edges, EPS_edges)

    # Annotate values in cells
    for ie in range(n_eps):
        for ig in range(n_g):
            val = Z[ie, ig]
            if not np.isnan(val):
                ax.text(g_vals[ig], eps_vals[ie], f'{val:.2f}',
                        ha='center', va='center', fontsize=6,
                        color='white' if val < np.nanmedian(valid) else 'black')

    ax.set_xticks(g_vals)
    ax.set_yticks(eps_vals)
    ax.tick_params(labelsize=7)
    ax.set_title(mlabel, fontsize=9)
    ax.set_xlabel('g', fontsize=8)
    ax.set_ylabel('ε', fontsize=8)
    fig.colorbar(pcm, ax=ax, fraction=0.046, pad=0.04)

legend_patch = mpatches.Patch(
    facecolor='#d0d0d0', edgecolor='#888', hatch='///',
    label='No activity (fr = 0 Hz — below threshold)')
fig.legend(handles=[legend_patch], loc='lower center',
           fontsize=10, ncol=1, bbox_to_anchor=(0.5, -0.02),
           frameon=True, fancybox=True)
plt.tight_layout()
plt.show()

## 7. Activity Summary Table
How many parameter combos produced activity per (N, J) slice?

In [ ]:
print(f'{"N":>6s} {"J":>6s}  active/total  % active')
print('-' * 42)
for N in N_vals:
    for J in J_vals:
        total = n_g * n_eps
        active = total - silent_mask_2d(N, J).sum()
        pct = 100 * active / total
        bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
        print(f'{int(N):>6d} {J:>6.2f}  {active:>3d}/{total:<3d}       {pct:5.1f}%  {bar}')